# Case Hit Probability — citation percentile within a cohort

A raw citation count is not comparable across centuries: a 1900 case has had 120 years to
accumulate citations and a 2015 case has had five, and a state trial court is not cited on the
scale of a supreme court. This ranks each case **inside its own cohort** so "top 1%" means the
same thing everywhere.

## The cohort

`(jurisdiction, decision_year)` — the case law analogue of `patent_hit_probability`'s
`(wipo_sector, grant_year)`. Jurisdiction is the closest thing here to a field: 61 of them, and
citation practice differs sharply between them. Year removes the accumulation advantage.

`pctl_C_w = percent_rank()` over `C_w` within the cohort, in [0, 1], so 0.99 is "cited more than
99% of cases from the same jurisdiction and year". Ties share the lowest rank, which matters a
great deal here: **1,391,837 cases are never cited**, so in most cohorts a large block sits at
percentile 0.

## Output
`Case law/output/case_hit_probability.parquet`

`case_id, jurisdiction, decision_year, cohort_n` and `pctl_C_{3,5,10,all}`.

`cohort_n` rides along because a percentile from a 6-case cohort is not the same evidence as one
from 40,000 — filter on it rather than trusting every row equally.

In [1]:
%%time
import os, sys
import numpy as np, pandas as pd, duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Case law')
import cl_common as cl
OUT_FP = cl.out('case_hit_probability.parquet')
CIT, META = cl.out('case_citation.parquet'), cl.out('case_metadata.parquet')
for f in (CIT, META):
    assert os.path.exists(f), f'{f} missing — run case_citation / case_metadata first'

con = duckdb.connect()
con.execute("SET memory_limit='24GB'")
con.execute(f"SET temp_directory='{cl.CACHE}/duckdb_tmp'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET enable_progress_bar=false")
con.execute(f"""
COPY (
  WITH j AS (
    SELECT c.case_id, m.jurisdiction, m.decision_year, c.C_3, c.C_5, c.C_10, c.C_all
    FROM read_parquet('{CIT}') c
    JOIN read_parquet('{META}') m USING (case_id)
    -- cl.YEAR_MIN floor: pre-1800 cohorts hold a handful of cases each, so a percentile
    -- from one is noise. The graph behind C_* is untouched.
    WHERE m.decision_year >= {cl.YEAR_MIN} AND m.jurisdiction IS NOT NULL
  )
  SELECT case_id, jurisdiction, decision_year,
         count(*)      OVER w                      AS cohort_n,
         percent_rank() OVER (PARTITION BY jurisdiction, decision_year ORDER BY C_3)   AS pctl_C_3,
         percent_rank() OVER (PARTITION BY jurisdiction, decision_year ORDER BY C_5)   AS pctl_C_5,
         percent_rank() OVER (PARTITION BY jurisdiction, decision_year ORDER BY C_10)  AS pctl_C_10,
         percent_rank() OVER (PARTITION BY jurisdiction, decision_year ORDER BY C_all) AS pctl_C_all
  FROM j
  WINDOW w AS (PARTITION BY jurisdiction, decision_year)
  ORDER BY case_id
) TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)
""")
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e6:.0f} MB)')

WROTE /project/jevans/Dawoon/Science of Science/Case law/output/case_hit_probability.parquet  (46 MB)


In [2]:
%%time
print(con.execute(f"""SELECT count(*) AS cases, count(DISTINCT jurisdiction) AS jurisdictions,
  min(decision_year) AS yr_min, max(decision_year) AS yr_max,
  round(avg(cohort_n),1) AS avg_cohort_n, min(cohort_n) AS min_cohort_n, max(cohort_n) AS max_cohort_n
  FROM read_parquet('{OUT_FP}')""").fetchdf().to_string(index=False))
print('\nshare of cases at percentile 0 (never cited, so tied at the bottom):')
print(con.execute(f"""SELECT round(100.0*count(*) FILTER (WHERE pctl_C_all = 0)/count(*),2) AS pct_at_zero_all,
  round(100.0*count(*) FILTER (WHERE pctl_C_10 = 0)/count(*),2) AS pct_at_zero_10
  FROM read_parquet('{OUT_FP}')""").fetchdf().to_string(index=False))
print('\ncohort sizes are wildly uneven — filter on cohort_n:')
display(con.execute(f"""SELECT jurisdiction, count(*) AS cases, round(avg(cohort_n),0) AS avg_cohort
  FROM read_parquet('{OUT_FP}') GROUP BY 1 ORDER BY cases DESC LIMIT 10""").fetchdf())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') LIMIT 5").fetchdf())

  cases  jurisdictions  yr_min  yr_max  avg_cohort_n  min_cohort_n  max_cohort_n
5177903             61    1800    2020        6462.8             1         37200

share of cases at percentile 0 (never cited, so tied at the bottom):
 pct_at_zero_all  pct_at_zero_10
           26.92           35.82

cohort sizes are wildly uneven — filter on cohort_n:


,jurisdiction,cases,avg_cohort
0,U.S.,1364193,19346.0
1,N.Y.,669625,5918.0
2,Tex.,223580,1751.0
3,Fla.,214970,3310.0
4,La.,188231,1887.0
5,Pa.,177510,1318.0
6,Ga.,153316,1248.0
7,Ill.,150521,1225.0
8,Cal.,138050,994.0
9,Mo.,116455,874.0


,case_id,jurisdiction,decision_year,cohort_n,pctl_C_3,pctl_C_5,pctl_C_10,pctl_C_all
0,12129810,U.S.,2016,28141,0.000000,0.000000,0.000000,0.000000
1,12129811,U.S.,2016,28141,0.695096,0.695025,0.695025,0.695025
2,12129812,U.S.,2016,28141,0.928465,0.928358,0.928358,0.928358
3,12129813,Ga.,2008,1707,0.572685,0.680539,0.652989,0.643611
4,12129814,W. Va.,2013,160,0.345912,0.484277,0.459119,0.459119
